# LeetCode 2000 — Reverse Prefix of Word bằng Python

**Link đề:** https://leetcode.com/problems/reverse-prefix-of-word/
**Độ khó:** Easy · **Chủ đề:** String, Two Pointers, Stack

```python
class Solution:
    def reversePrefix(self, word: str, ch: str) -> str:

```

## Mục tiêu học
1. Hiểu đề: đảo đoạn `word[0..i]` với `i` là vị trí **xuất hiện đầu tiên** của `ch`; không có `ch` thì giữ nguyên.
2. Nắm công cụ chuỗi: `str.find`, `str.index`, slicing `s[:i+1]`, `s[::-1]`, ghép chuỗi `+`, tính **bất biến** (immutable) của `str`.
3. Viết lời giải ngắn bằng slicing, hiểu chính xác vì sao `+1`.
4. Học kỹ thuật **hai con trỏ** đảo tại chỗ trên `list`, và cách dùng **stack** — trace từng bước.
5. So sánh 4 cách, test biên (ch ở đầu, ở cuối, không có, xuất hiện nhiều lần), làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho chuỗi `word` (chỉ số từ 0) và ký tự `ch`. Đảo ngược đoạn của `word` **bắt đầu từ chỉ số 0** và **kết thúc tại chỉ số xuất hiện đầu tiên của `ch`** (tính cả vị trí đó). Nếu `ch` không có trong `word`, không làm gì.

| `word` | `ch` | Vị trí đầu tiên | Kết quả |
|---|:-:|:-:|---|
| `"abcdefd"` | `d` | 3 | `"dcba" + "efd"` = `"dcbaefd"` |
| `"xyxzxe"` | `z` | 3 | `"zxyx" + "xe"` = `"zxyxxe"` |
| `"abcd"` | `z` | không có | `"abcd"` |

**Constraints:** `1 <= word.length <= 250`, `word` và `ch` là chữ thường tiếng Anh.

Chú ý ví dụ 1: `d` xuất hiện **hai lần** (vị trí 3 và 6) — chỉ dùng lần **đầu tiên**.

In [1]:
word, ch = "abcdefd", "d"
for i, c in enumerate(word):
    mark = "<- ch" if c == ch else ""
    print(i, c, mark)

0 a 
1 b 
2 c 
3 d <- ch
4 e 
5 f 
6 d <- ch


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi là **bất biến** (immutable)

Không thể gán `word[0] = 'x'`. Muốn "sửa" chuỗi phải **tạo chuỗi mới** (bằng slicing + ghép), hoặc đổi sang `list` rồi sửa, cuối cùng `"".join(...)`.

In [2]:
word = "abc"
try:
    word[0] = "x"
except TypeError as e:
    print("Lỗi:", e)

chars = list(word)        # ['a', 'b', 'c'] — list thì sửa được
chars[0] = "x"
print(chars, "->", "".join(chars))

Lỗi: 'str' object does not support item assignment
['x', 'b', 'c'] -> xbc


### 2.2. `str.find` và `str.index`

- `word.find(ch)` trả về chỉ số **đầu tiên** của `ch`, hoặc **`-1`** nếu không có.
- `word.index(ch)` giống `find` nhưng **ném `ValueError`** nếu không có.

Bài này `find` tiện hơn vì xử lý "không có" bằng `-1`.

In [3]:
print("abcdefd".find("d"))    # 3 — lần đầu, không phải 6
print("abcd".find("z"))       # -1
print("abcdefd".index("d"))   # 3
try:
    "abcd".index("z")
except ValueError as e:
    print("index ném lỗi:", e)

3
-1
3
index ném lỗi: substring not found


### 2.3. Slicing `s[:i+1]`, `s[i+1:]` và `s[::-1]`

- `s[:i+1]` = các ký tự chỉ số `0..i` (**tính cả** `i`) — vì `stop` không được lấy nên phải `+1`.
- `s[i+1:]` = phần còn lại sau `i`.
- `s[::-1]` = chuỗi đảo ngược (bước `-1`).

In [4]:
s, i = "abcdefd", 3
print("s[:i+1] =", s[:i + 1])
print("s[:i]   =", s[:i], " <- thiếu ký tự 'd' ở vị trí i")
print("s[i+1:] =", s[i + 1:])
print("đảo     =", s[:i + 1][::-1])

s[:i+1] = abcd
s[:i]   = abc  <- thiếu ký tự 'd' ở vị trí i
s[i+1:] = efd
đảo     = dcba


### 2.4. Slicing ngược có `start`/`stop`: `s[i::-1]`

`s[i::-1]` bắt đầu ở `i`, đi lùi tới đầu chuỗi → chính là `s[:i+1][::-1]` nhưng viết một bước.

In [5]:
s, i = "abcdefd", 3
print(s[i::-1])
print(s[i::-1] == s[:i + 1][::-1])

dcba
True


### 2.5. Hoán đổi hai biến / hai phần tử list

Python cho phép `a, b = b, a`. Với list: `arr[l], arr[r] = arr[r], arr[l]` — đây là "nguyên tử" của kỹ thuật hai con trỏ.

In [6]:
arr = list("abcd")
arr[0], arr[3] = arr[3], arr[0]
print(arr)

['d', 'b', 'c', 'a']


## 3. Cách 1 — Trực giác: tìm vị trí, cắt, đảo, ghép

Ba bước (đúng như hai hint của đề):
1. `i = word.find(ch)`.
2. Nếu `i == -1` → trả `word`.
3. Ngược lại trả `word[:i+1][::-1] + word[i+1:]`.

In [7]:
def reverse_prefix_slice(word, ch):
    i = word.find(ch)
    if i == -1:
        return word
    return word[:i + 1][::-1] + word[i + 1:]

print(reverse_prefix_slice("abcdefd", "d"))   # dcbaefd
print(reverse_prefix_slice("xyxzxe", "z"))    # zxyxxe
print(reverse_prefix_slice("abcd", "z"))      # abcd

dcbaefd
zxyxxe
abcd


### Điều thú vị: không cần `if` cũng đúng?

Nếu `i = -1` thì `word[:0][::-1] + word[0:]` = `"" + word` = `word`. Vậy công thức tự đúng khi không có `ch`! Nhưng viết `if` rõ ràng giúp người đọc hiểu ý đồ — nên giữ.

In [8]:
word, i = "abcd", -1
print(repr(word[:i + 1]), "+", repr(word[i + 1:]), "=", word[:i + 1][::-1] + word[i + 1:])

'' + 'abcd' = abcd


## 4. Cách 2 — Tự tìm vị trí bằng vòng lặp (không dùng `find`)

Để hiểu `find` làm gì bên trong: duyệt từ trái, gặp `ch` lần đầu thì **dừng ngay** (`break`). Dùng biến `pos = -1` làm giá trị mặc định "chưa thấy".

In [9]:
def first_index(word, ch):
    pos = -1
    for i, c in enumerate(word):
        if c == ch:
            pos = i
            break          # dừng ở lần đầu tiên
    return pos

def reverse_prefix_loop(word, ch):
    pos = first_index(word, ch)
    if pos == -1:
        return word
    return word[pos::-1] + word[pos + 1:]

print(first_index("abcdefd", "d"), first_index("abcd", "z"))
print(reverse_prefix_loop("abcdefd", "d"))

3 -1
dcbaefd


## 5. Nút thắt / câu hỏi phỏng vấn: nếu không được dùng slicing đảo?

Slicing rất tiện, nhưng phỏng vấn viên hay hỏi: *"Đảo tại chỗ (in-place) không tạo chuỗi phụ thì sao?"* — trong các ngôn ngữ như C++/Java, chuỗi/mảng ký tự thường được đảo bằng **hai con trỏ**.

**Insight hai con trỏ:** đảo đoạn `[0..i]` = hoán đổi cặp đối xứng `(0, i)`, `(1, i−1)`, ... cho tới khi hai con trỏ gặp nhau. Số lần hoán đổi ≈ `(i+1)/2`.

| Bước | left | right | Trạng thái `abcd|efd` |
|:-:|:-:|:-:|---|
| 0 | 0 | 3 | `a b c d` |
| 1 | 1 | 2 | `d b c a` (đổi 0↔3) |
| 2 | 2 | 1 | `d c b a` (đổi 1↔2) → dừng vì left ≥ right |

## 6. Cách 3 — Hai con trỏ trên list (lời giải chính để phỏng vấn)

Giảng từng đoạn:

```python
i = word.find(ch)                 # (1) vị trí kết thúc đoạn cần đảo
if i == -1:
    return word                   # (2) không có ch
chars = list(word)                # (3) chuỗi bất biến -> đổi sang list
left, right = 0, i                # (4) hai đầu đoạn
while left < right:               # (5) còn cặp chưa đổi
    chars[left], chars[right] = chars[right], chars[left]
    left += 1
    right -= 1
return "".join(chars)             # (6) ghép lại thành chuỗi
```

In [10]:
def reverse_prefix_two_pointers(word, ch):
    i = word.find(ch)
    if i == -1:
        return word
    chars = list(word)
    left, right = 0, i
    while left < right:
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

print(reverse_prefix_two_pointers("abcdefd", "d"))
print(reverse_prefix_two_pointers("xyxzxe", "z"))
print(reverse_prefix_two_pointers("abcd", "z"))

dcbaefd
zxyxxe
abcd


## 7. TRACE từng bước (hai con trỏ)

Ví dụ `word = "xyxzxe"`, `ch = "z"` → `i = 3`. Ta in `chars`, `left`, `right` **trước** mỗi lần hoán đổi.

In [11]:
def trace_reverse_prefix(word, ch):
    i = word.find(ch)
    print(f"word = {word!r}, ch = {ch!r} -> i = {i}")
    if i == -1:
        print("  không có ch -> giữ nguyên")
        return word
    chars = list(word)
    left, right = 0, i
    step = 1
    while left < right:
        print(f"  Bước {step}: left={left} ({chars[left]}), right={right} ({chars[right]}) -> đổi chỗ")
        chars[left], chars[right] = chars[right], chars[left]
        print(f"           chars = {''.join(chars)}")
        left += 1
        right -= 1
        step += 1
    print(f"  Dừng: left={left}, right={right} (left >= right)")
    res = "".join(chars)
    print("  Kết quả:", res)
    return res

trace_reverse_prefix("xyxzxe", "z")
print()
trace_reverse_prefix("abcdefd", "d")

word = 'xyxzxe', ch = 'z' -> i = 3
  Bước 1: left=0 (x), right=3 (z) -> đổi chỗ
           chars = zyxxxe
  Bước 2: left=1 (y), right=2 (x) -> đổi chỗ
           chars = zxyxxe
  Dừng: left=2, right=1 (left >= right)
  Kết quả: zxyxxe

word = 'abcdefd', ch = 'd' -> i = 3
  Bước 1: left=0 (a), right=3 (d) -> đổi chỗ
           chars = dbcaefd
  Bước 2: left=1 (b), right=2 (c) -> đổi chỗ
           chars = dcbaefd
  Dừng: left=2, right=1 (left >= right)
  Kết quả: dcbaefd


'dcbaefd'

### Nhánh đặc biệt

- `ch` ở **vị trí 0**: `left = right = 0` → vòng `while` không chạy → chuỗi không đổi (đảo 1 ký tự là chính nó).
- Đoạn có số ký tự **lẻ** (ví dụ `i = 4`): ký tự giữa (`left == right == 2`) không cần đổi.

In [12]:
trace_reverse_prefix("abc", "a")
print()
trace_reverse_prefix("abcdez", "e")

word = 'abc', ch = 'a' -> i = 0
  Dừng: left=0, right=0 (left >= right)
  Kết quả: abc

word = 'abcdez', ch = 'e' -> i = 4
  Bước 1: left=0 (a), right=4 (e) -> đổi chỗ
           chars = ebcdaz
  Bước 2: left=1 (b), right=3 (d) -> đổi chỗ
           chars = edcbaz
  Dừng: left=2, right=2 (left >= right)
  Kết quả: edcbaz


'edcbaz'

## 8. Code nộp LeetCode

Bản slicing (ngắn, idiomatic Python) và bản hai con trỏ (thể hiện kỹ thuật) đều được Accept.

In [13]:
class Solution:
    def reversePrefix(self, word: str, ch: str) -> str:
        i = word.find(ch)
        if i == -1:
            return word
        return word[i::-1] + word[i + 1:]


class SolutionTwoPointers:
    def reversePrefix(self, word: str, ch: str) -> str:
        i = word.find(ch)
        if i == -1:
            return word
        chars = list(word)
        left, right = 0, i
        while left < right:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
        return "".join(chars)


for w, c, expected in [("abcdefd", "d", "dcbaefd"), ("xyxzxe", "z", "zxyxxe"), ("abcd", "z", "abcd")]:
    assert Solution().reversePrefix(w, c) == expected
    assert SolutionTwoPointers().reversePrefix(w, c) == expected
print("Hai bản nộp đều đúng 3 ví dụ của đề")

Hai bản nộp đều đúng 3 ví dụ của đề


## 9. Cách 4 — Stack (ngăn xếp), một lần duyệt

Hint gợi ý chủ đề **Stack**: stack là cấu trúc **vào sau ra trước (LIFO)**. Đẩy các ký tự vào stack rồi lấy ra sẽ được thứ tự **ngược**.

Ý tưởng một lần duyệt: đẩy ký tự vào stack cho tới khi gặp `ch` (đẩy luôn `ch`). Khi gặp, **pop hết** stack (ra thứ tự đảo) rồi nối phần còn lại. Nếu duyệt hết mà không gặp `ch` → trả `word`.

Trong Python, `list` dùng như stack: `append` = push, `pop()` = lấy phần tử cuối.

In [14]:
stack = []
for c in "abc":
    stack.append(c)
    print("push", c, "->", stack)
out = []
while stack:
    out.append(stack.pop())
print("pop hết ->", "".join(out))

push a -> ['a']
push b -> ['a', 'b']
push c -> ['a', 'b', 'c']
pop hết -> cba


In [15]:
def reverse_prefix_stack(word, ch):
    stack = []
    for i, c in enumerate(word):
        stack.append(c)
        if c == ch:
            rev = []
            while stack:
                rev.append(stack.pop())
            return "".join(rev) + word[i + 1:]
    return word

print(reverse_prefix_stack("abcdefd", "d"))
print(reverse_prefix_stack("xyxzxe", "z"))
print(reverse_prefix_stack("abcd", "z"))

dcbaefd
zxyxxe
abcd


## 10. Vì sao đúng?

**Cách 1:** `word = word[:i+1] + word[i+1:]` (chia đúng tại `i`, tính cả `i` vào phần đầu). Đảo phần đầu và giữ phần sau đúng theo định nghĩa đề. `find` trả về **lần đầu** nên `i` đúng là vị trí đầu tiên.

**Cách 3 — invariant hai con trỏ:** trước mỗi vòng, các ký tự ở vị trí `< left` và `> right` (trong đoạn `[0..i]`) **đã nằm đúng chỗ của chuỗi đảo**, còn đoạn `[left..right]` chưa đụng tới. Mỗi vòng đổi `chars[left] ↔ chars[right]` thì hai vị trí này cũng đúng chỗ, rồi thu hẹp đoạn. Khi `left ≥ right` đoạn chưa xử lý có ≤ 1 ký tự (ký tự giữa đứng yên) → toàn đoạn đã đảo. Phần `> i` không bị chạm.

**Cách 4:** stack là LIFO nên `pop` lần lượt trả `word[i], word[i-1], ..., word[0]` — đúng là đoạn đảo.

## 11. Time / Space và bảng so sánh

Gọi `n = len(word)`, `i` = vị trí đầu tiên của `ch`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. `find` + slicing | `word[:i+1][::-1] + word[i+1:]` | `O(n)` | `O(n)` | Ngắn nhất, **khuyến nghị cho Python** |
| 2. Vòng lặp tìm + slicing | như 1, tự viết `find` | `O(n)` | `O(n)` | Hiểu bản chất `find` |
| 3. Hai con trỏ | đổi chỗ trên list | `O(n)` | `O(n)` (list + chuỗi kết quả) | Kỹ thuật phỏng vấn |
| 4. Stack | push rồi pop | `O(n)` | `O(n)` | Một lần duyệt |

Vì `str` của Python bất biến, mọi cách đều phải tạo chuỗi kết quả mới → `O(n)` bộ nhớ là không tránh được. Phần "tại chỗ" của cách 3 chỉ có nghĩa trên `list`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. `ch` ở vị trí 0 → không đổi.
3. `ch` ở **cuối** → đảo cả chuỗi.
4. `ch` xuất hiện nhiều lần → chỉ đến lần đầu.
5. Chuỗi 1 ký tự (có và không có `ch`).
6. Đối chiếu ngẫu nhiên giữa 4 cách và một "oracle" viết theo định nghĩa.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS ({actual!r})")

f = reverse_prefix_slice
check("đề 1", f("abcdefd", "d"), "dcbaefd")
check("đề 2", f("xyxzxe", "z"), "zxyxxe")
check("đề 3", f("abcd", "z"), "abcd")
check("ch ở đầu", f("abcd", "a"), "abcd")
check("ch ở cuối", f("abcd", "d"), "dcba")
check("ch lặp lại", f("abab", "b"), "baab")
check("1 ký tự có ch", f("a", "a"), "a")
check("1 ký tự không có ch", f("a", "b"), "a")

đề 1                       PASS ('dcbaefd')
đề 2                       PASS ('zxyxxe')
đề 3                       PASS ('abcd')
ch ở đầu                   PASS ('abcd')
ch ở cuối                  PASS ('dcba')
ch lặp lại                 PASS ('baab')
1 ký tự có ch              PASS ('a')
1 ký tự không có ch        PASS ('a')


In [17]:
import random

def oracle(word, ch):
    if ch not in word:
        return word
    i = min(k for k in range(len(word)) if word[k] == ch)
    return "".join(reversed(word[:i + 1])) + word[i + 1:]

rng = random.Random(2000)
for _ in range(3000):
    word = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 12)))
    ch = rng.choice("abcde")
    exp = oracle(word, ch)
    for fn in (reverse_prefix_slice, reverse_prefix_loop, reverse_prefix_two_pointers, reverse_prefix_stack):
        assert fn(word, ch) == exp, (fn.__name__, word, ch)
    assert Solution().reversePrefix(word, ch) == exp
print("3000 test ngẫu nhiên: mọi cách khớp oracle")

3000 test ngẫu nhiên: mọi cách khớp oracle


`reversed(s)` trả về một **iterator** duyệt ngược; cần `"".join(...)` để ghép lại thành chuỗi. Ta dùng nó trong oracle để có cách viết **khác** các lời giải (tránh oracle mắc cùng lỗi với lời giải).

## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên `+1`, đảo thiếu ký tự `ch`

In [18]:
def wrong_no_plus_one(word, ch):
    i = word.find(ch)
    if i == -1:
        return word
    return word[:i][::-1] + word[i:]

print("sai :", wrong_no_plus_one("abcdefd", "d"))   # cbadefd
print("đúng:", reverse_prefix_slice("abcdefd", "d"))

sai : cbadefd
đúng: dcbaefd


**Vì sao sai:** đề nói "**inclusive**" — đoạn đảo gồm cả vị trí `i`. `word[:i]` dừng trước `i`. **Sửa:** `word[:i+1]`.

### Lỗi 2: Dùng `rfind` / tìm lần xuất hiện **cuối**

In [19]:
def wrong_last(word, ch):
    i = word.rfind(ch)              # rfind: tìm từ phải
    if i == -1:
        return word
    return word[:i + 1][::-1] + word[i + 1:]

print("sai :", wrong_last("abcdefd", "d"))   # dfedcba
print("đúng:", reverse_prefix_slice("abcdefd", "d"))

sai : dfedcba
đúng: dcbaefd


**Vì sao sai:** đề yêu cầu **first occurrence**. **Sửa:** `find` (từ trái) hoặc `break` ngay lần đầu.

### Lỗi 3: Dùng `index` mà không xử lý trường hợp không có `ch`

In [20]:
def wrong_index(word, ch):
    i = word.index(ch)              # ném ValueError nếu không có
    return word[:i + 1][::-1] + word[i + 1:]

try:
    wrong_index("abcd", "z")
except ValueError as e:
    print("sai : ValueError ->", e)
print("đúng:", reverse_prefix_slice("abcd", "z"))

sai : ValueError -> substring not found
đúng: abcd


**Sửa:** dùng `find` (trả `-1`), hoặc kiểm tra `if ch not in word: return word` trước.

### Lỗi 4: Hai con trỏ với điều kiện `while left <= right` hoặc chạy hết `range(i+1)`

In [21]:
def wrong_swap_all(word, ch):
    i = word.find(ch)
    chars = list(word)
    for k in range(i + 1):                    # đổi cả nửa sau -> đổi về như cũ!
        chars[k], chars[i - k] = chars[i - k], chars[k]
    return "".join(chars)

print("sai :", wrong_swap_all("abcdefd", "d"))   # abcdefd — không đổi gì
print("đúng:", reverse_prefix_slice("abcdefd", "d"))

sai : abcdefd
đúng: dcbaefd


**Vì sao sai:** mỗi cặp bị đổi **hai lần** (một lần ở nửa đầu, một lần ở nửa sau) → trở về ban đầu. **Sửa:** chỉ đổi khi `left < right` (hoặc `for k in range((i + 1) // 2)`). Riêng `left <= right` thì vô hại (đổi ký tự giữa với chính nó) nhưng thừa một bước.

## 14. Mẹo nhận dạng pattern

- "Đảo một đoạn của chuỗi/mảng" → slicing `[::-1]` trong Python, hoặc **hai con trỏ** từ hai đầu đi vào.
- "First occurrence" → `find` / `index` / vòng lặp + `break`. "Last occurrence" → `rfind`.
- Chuỗi bất biến → muốn sửa nhiều chỗ: `list(s)` → sửa → `"".join(...)`.
- Kết quả cần "thứ tự ngược với thứ tự đọc" → nghĩ tới **stack**.

## 15. Biến thể

1. **LeetCode 344 — Reverse String:** đảo cả list ký tự tại chỗ (hai con trỏ).
2. **LeetCode 541 — Reverse String II:** cứ mỗi `2k` ký tự, đảo `k` ký tự đầu.
3. **Đảo đoạn `[l, r]` bất kỳ.**
4. **LeetCode 917 — Reverse Only Letters:** hai con trỏ bỏ qua ký tự không phải chữ.

In [22]:
def reverse_range(s, l, r):
    return s[:l] + s[l:r + 1][::-1] + s[r + 1:]

def reverse_str_ii(s, k):
    chars = list(s)
    for start in range(0, len(s), 2 * k):
        chars[start:start + k] = reversed(chars[start:start + k])   # gán vào slice của list
    return "".join(chars)

print(reverse_range("abcdefg", 2, 4))      # abedcfg
print(reverse_str_ii("abcdefg", 2))        # bacdfeg

abedcfg
bacdfeg


`chars[a:b] = ...` là **gán vào slice**: thay đoạn `a..b-1` của list bằng các phần tử mới — chỉ làm được với `list`, không với `str`.

## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
`reversePrefix("leetcode", "e")` = ? Ghi rõ `i`, phần đầu, phần sau.

### Bài 2 — Dự đoán output
`"hello"[3::-1]` và `"hello"[:3][::-1]` in ra gì? Có bằng nhau không?

### Bài 3 — Tự viết lời giải bằng slicing

In [23]:
def exercise_reverse_prefix(word: str, ch: str) -> str:
    # TODO: i = word.find(ch); xử lý -1; ghép phần đảo + phần sau
    pass

# assert exercise_reverse_prefix("abcdefd", "d") == "dcbaefd"
# assert exercise_reverse_prefix("abcd", "z") == "abcd"

### Bài 4 — Tự viết hàm đảo list ký tự tại chỗ (LeetCode 344), không dùng `[::-1]` hay `reverse()`

In [24]:
def exercise_reverse_in_place(chars: list[str]) -> None:
    # TODO: hai con trỏ left, right; while left < right: đổi chỗ
    pass

# arr = list("hello"); exercise_reverse_in_place(arr); assert arr == list("olleh")

### Bài 5 — Biến thể: đảo **hậu tố** bắt đầu từ lần xuất hiện **cuối** của `ch` tới hết chuỗi

In [25]:
def exercise_reverse_suffix(word: str, ch: str) -> str:
    # TODO: i = word.rfind(ch); đảo word[i:]
    pass

# assert exercise_reverse_suffix("abcdefd", "b") == "adfedcb"
# assert exercise_reverse_suffix("abc", "z") == "abc"

### Bài 6 — Mini challenge: LeetCode 541 — Reverse String II

In [26]:
def exercise_reverse_str_ii(s: str, k: int) -> str:
    # TODO: với mỗi start = 0, 2k, 4k, ...: đảo s[start:start+k]
    pass

# assert exercise_reverse_str_ii("abcdefg", 2) == "bacdfeg"
# assert exercise_reverse_str_ii("abcd", 4) == "dcba"

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `"leetcode".find("e")` | `i = 1` | `"el" + "etcode"` |
| 2 | `[3::-1]` bắt đầu tại 3 đi lùi | chỉ số 3 là `'l'` | cả hai: `"lleh"` vs `"leh"`? tính kỹ! |
| 3 | Khuôn mục 3 | `word[:i + 1][::-1]` | `+ word[i + 1:]` |
| 4 | `left, right = 0, len(chars) - 1` | `while left < right` | đổi chỗ, `left += 1`, `right -= 1` |
| 5 | Dùng `rfind` | phần đầu `word[:i]` giữ nguyên | `word[:i] + word[i:][::-1]` |
| 6 | `range(0, len(s), 2 * k)` | list + gán slice | `"".join(chars)` |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: `i = 1`; phần đầu `"le"` → đảo `"el"`; phần sau `"etcode"` → **`"eletcode"`**.
- Bài 2: `"hello"[3::-1]` = **`"lleh"`** (chỉ số 3,2,1,0); `"hello"[:3][::-1]` = **`"leh"`** (chỉ số 2,1,0) → **khác nhau**. `[i::-1]` tương ứng với `[:i+1][::-1]`, không phải `[:i][::-1]`.

In [27]:
def answer_reverse_prefix(word, ch):
    i = word.find(ch)
    return word if i == -1 else word[:i + 1][::-1] + word[i + 1:]

def answer_reverse_in_place(chars):
    left, right = 0, len(chars) - 1
    while left < right:
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1

def answer_reverse_suffix(word, ch):
    i = word.rfind(ch)
    return word if i == -1 else word[:i] + word[i:][::-1]

def answer_reverse_str_ii(s, k):
    chars = list(s)
    for start in range(0, len(s), 2 * k):
        chars[start:start + k] = chars[start:start + k][::-1]
    return "".join(chars)

assert answer_reverse_prefix("leetcode", "e") == "eletcode"
assert "hello"[3::-1] == "lleh" and "hello"[:3][::-1] == "leh"
assert answer_reverse_prefix("abcdefd", "d") == "dcbaefd" and answer_reverse_prefix("abcd", "z") == "abcd"
arr = list("hello"); answer_reverse_in_place(arr); assert arr == list("olleh")
arr = list("ab"); answer_reverse_in_place(arr); assert arr == list("ba")
assert answer_reverse_suffix("abcdefd", "b") == "adfedcb" and answer_reverse_suffix("abc", "z") == "abc"
assert answer_reverse_str_ii("abcdefg", 2) == "bacdfeg" and answer_reverse_str_ii("abcd", 4) == "dcba"
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def reversePrefix(self, word: str, ch: str) -> str:
        i = word.find(ch)                       # -1 nếu không có
        if i == -1:
            return word
        return word[:i + 1][::-1] + word[i + 1:]   # inclusive -> +1
```

Hai con trỏ: `while l < r: a[l], a[r] = a[r], a[l]; l += 1; r -= 1`.

### Checklist trước khi submit
- [ ] Dùng lần xuất hiện **đầu tiên** (`find`, không `rfind`)?
- [ ] Đoạn đảo **tính cả** vị trí `i` (`[:i+1]`)?
- [ ] Xử lý `ch` không có (`-1`) mà không ném lỗi?
- [ ] Hai con trỏ dừng khi `left >= right` (không đổi mỗi cặp hai lần)?
- [ ] Đã test `ch` ở đầu, ở cuối và xuất hiện nhiều lần?